# Gradient Checkpointing

Wiki reference for [gradient checkpointing](https://ml-viz-ruby.vercel.app/wiki/gradient-checkpointing).

**The idea in one sentence.** Backprop normally caches **every** layer's activations (memory
$\propto$ depth $L$); gradient checkpointing instead stores only $\sim\sqrt{L}$ **checkpoints**
and **recomputes** the rest during the backward pass — cutting activation memory from $O(L)$ to
$O(\sqrt{L})$ at the cost of one extra forward pass.

We implement full-cache and checkpointed backprop from scratch, **validate that the gradients
are identical and that peak memory drops to $\sim\sqrt{L}$**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — A toy L-layer MLP

Every layer is the same shape (`d -> d`) so each activation costs exactly 1 "unit" of memory. We track how many activation units are alive at any point in time, rather than the literal byte count — that's all that matters for the $O(L)$ vs $O(\sqrt{L})$ comparison.

In [ ]:
d = 32  # hidden width -- irrelevant to the memory-unit counting, just needs to be fixed

def make_mlp(L, seed=0):
    r = np.random.default_rng(seed)
    Ws = [r.normal(scale=1.0/np.sqrt(d), size=(d, d)) for _ in range(L)]
    bs = [np.zeros(d) for _ in range(L)]
    return Ws, bs

def relu(x):
    return np.maximum(x, 0.0)

def relu_grad(x):
    return (x > 0).astype(x.dtype)

def layer_forward(a_prev, W, b):
    z = a_prev @ W + b
    a = relu(z)
    return a, z

def layer_backward(grad_a, z, a_prev, W):
    grad_z = grad_a * relu_grad(z)
    grad_W = np.outer(a_prev, grad_z)
    grad_b = grad_z
    grad_a_prev = grad_z @ W.T
    return grad_a_prev, grad_W, grad_b

## 2 — No checkpointing: cache every activation

The naive implementation stores every layer's *input* activation `a_0 .. a_{L-1}` during the forward pass (needed by `layer_backward`), then walks backward through them. Peak memory = `L` activation units, and we track the peak explicitly with a simple counter.

In [ ]:
def forward_backward_no_checkpoint(x, Ws, bs, grad_out):
    L = len(Ws)
    cache_a, cache_z = [], []
    a = x
    peak_units = 0
    for i in range(L):
        cache_a.append(a)           # store input activation -- +1 unit, kept until backward consumes it
        a, z = layer_forward(a, Ws[i], bs[i])
        cache_z.append(z)
        peak_units = max(peak_units, len(cache_a))
    # backward: pops from the end, but nothing is freed until the whole pass starts consuming
    peak_units = L  # every a_0..a_{L-1} is alive simultaneously right before backward begins
    grad_a = grad_out
    grad_Ws, grad_bs = [None]*L, [None]*L
    for i in reversed(range(L)):
        grad_a, grad_W, grad_b = layer_backward(grad_a, cache_z[i], cache_a[i], Ws[i])
        grad_Ws[i], grad_bs[i] = grad_W, grad_b
    return grad_a, grad_Ws, grad_bs, peak_units

## 3 — With checkpointing: store every √L-th activation, recompute the rest

Following the wiki page's segment-boundary heuristic, we checkpoint every $m = \lceil\sqrt{L}\rceil$ layers. During the forward pass we discard intra-segment activations immediately. During the backward pass, for each segment (processed in reverse), we **replay** its forward pass from the nearest checkpoint to regenerate the activations we need, run backprop through just that segment, then discard the recomputed activations again.

In [ ]:
def forward_checkpointed(x, Ws, bs, m):
    """Forward pass storing only checkpoints every m layers. Returns checkpoints + final output."""
    L = len(Ws)
    checkpoints = [x]  # a_0
    a = x
    for i in range(L):
        a, _ = layer_forward(a, Ws[i], bs[i])
        if (i + 1) % m == 0 or i == L - 1:
            checkpoints.append(a)
    return checkpoints  # length ~= L/m + 1

def replay_segment(a_start, Ws, bs, start, end):
    """Recompute the forward pass for layers [start, end), caching every intermediate."""
    cache_a, cache_z = [], []
    a = a_start
    for i in range(start, end):
        cache_a.append(a)
        a, z = layer_forward(a, Ws[i], bs[i])
        cache_z.append(z)
    return cache_a, cache_z, a

def forward_backward_checkpointed(x, Ws, bs, grad_out, m):
    L = len(Ws)
    checkpoints = forward_checkpointed(x, Ws, bs, m)
    n_checkpoints = len(checkpoints)

    # segment boundaries: [0, m, 2m, ...], each segment covers layers [start, end)
    boundaries = list(range(0, L, m)) + [L]
    segments = list(zip(boundaries[:-1], boundaries[1:]))

    grad_a = grad_out
    grad_Ws, grad_bs = [None]*L, [None]*L
    peak_units = n_checkpoints  # checkpoints held throughout

    for seg_idx in reversed(range(len(segments))):
        start, end = segments[seg_idx]
        a_start = checkpoints[seg_idx]
        cache_a, cache_z, _ = replay_segment(a_start, Ws, bs, start, end)
        # peak = checkpoints still resident + this segment's recomputed activations
        peak_units = max(peak_units, n_checkpoints + len(cache_a))
        for i in reversed(range(start, end)):
            local = i - start
            grad_a, grad_W, grad_b = layer_backward(grad_a, cache_z[local], cache_a[local], Ws[i])
            grad_Ws[i], grad_bs[i] = grad_W, grad_b
        # recomputed activations for this segment are discarded here (out of scope)

    return grad_a, grad_Ws, grad_bs, peak_units

## 4 — Verify the gradients match exactly

Checkpointing must be numerically identical to the no-checkpoint pass -- it only changes *when* activations are computed, not the math.

In [ ]:
L = 16
Ws, bs = make_mlp(L, seed=1)
x = rng.normal(size=d)
grad_out = rng.normal(size=d)

_, gW_full, gb_full, peak_full = forward_backward_no_checkpoint(x, Ws, bs, grad_out)

m = int(np.ceil(np.sqrt(L)))
_, gW_ckpt, gb_ckpt, peak_ckpt = forward_backward_checkpointed(x, Ws, bs, grad_out, m)

max_diff = max(np.abs(a - b).max() for a, b in zip(gW_full, gW_ckpt))
print(f'L={L}, segment size m=sqrt(L)~{m}')
print(f'Max gradient difference (checkpointed vs full cache): {max_diff:.2e}')
print(f'Peak activation units -- no checkpoint: {peak_full}, checkpointed: {peak_ckpt}')
assert max_diff < 1e-10, 'checkpointed gradients must match the full-cache gradients exactly'

### Validate: checkpointing gives identical gradients with less memory

Recomputing activations must produce **exactly** the same gradients as caching them — it only
changes *when* activations exist, not their values. Meanwhile peak activation memory drops. We
confirm both.

In [ ]:
print(f'peak activation units: full={peak_full}, checkpointed={peak_ckpt}; gradient max diff={max_diff:.1e}')
assert max_diff < 1e-10, 'checkpointing recomputes activations -> identical gradients'
assert peak_ckpt < peak_full, 'checkpointing stores fewer activations -> lower peak memory'
print('\n✅ same gradients, less memory — checkpointing only trades memory for recompute')

## 5 — Memory vs depth: reproduce the O(√L) curve

Sweep `L` and compare peak activation memory (no checkpointing vs checkpointing with $m=\sqrt{L}$), matching the wiki page's worked table.

In [ ]:
Ls = [16, 64, 256, 1024]
no_ckpt_peaks, ckpt_peaks, ms = [], [], []
for L_ in Ls:
    Ws_, bs_ = make_mlp(L_, seed=2)
    x_ = rng.normal(size=d)
    go_ = rng.normal(size=d)
    _, _, _, p_full = forward_backward_no_checkpoint(x_, Ws_, bs_, go_)
    m_ = int(np.ceil(np.sqrt(L_)))
    _, _, _, p_ckpt = forward_backward_checkpointed(x_, Ws_, bs_, go_, m_)
    no_ckpt_peaks.append(p_full); ckpt_peaks.append(p_ckpt); ms.append(m_)
    print(f'L={L_:5d}  sqrt(L)~{m_:3d}  no-checkpoint peak={p_full:5d}  checkpointed peak={p_ckpt:4d}  reduction={p_full/p_ckpt:.1f}x')

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(Ls, no_ckpt_peaks, 'o-', color='#f87171', label='No checkpointing: O(L)')
ax[0].plot(Ls, ckpt_peaks, 'o-', color='#34d399', label='Checkpointed: O(sqrt(L))')
ax[0].set_xlabel('L (layers)'); ax[0].set_ylabel('peak activation units')
ax[0].set_title('Peak activation memory vs depth'); ax[0].legend()

reductions = [f/c for f, c in zip(no_ckpt_peaks, ckpt_peaks)]
ax[1].plot(Ls, reductions, 'o-', color='#6366f1')
ax[1].set_xlabel('L (layers)'); ax[1].set_ylabel('memory reduction factor')
ax[1].set_title('Reduction factor grows ~sqrt(L)/2 with depth')
plt.tight_layout(); plt.show()

### Validate: memory scales as $\sqrt{L}$, not $L$

Full caching's peak grows **linearly** with depth (peak $= L$); checkpointing's grows like
$\sqrt{L}$, so the gap widens with depth — exactly why it enables training very deep networks.
We confirm the linear vs sub-linear scaling.

In [ ]:
print('L          :', list(Ls))
print('full peak  :', list(no_ckpt_peaks))
print('ckpt peak  :', list(ckpt_peaks))
assert list(no_ckpt_peaks) == list(Ls), 'full-cache peak memory scales linearly with depth L'
assert ckpt_peaks[-1] < no_ckpt_peaks[-1] / 4, 'checkpointed peak grows ~sqrt(L) — far below L for deep nets'
print('\n✅ O(sqrt(L)) vs O(L) memory — the deeper the net, the bigger the saving')

## 6 — Extra forward compute: measure it, don't just assert it

Count the actual number of `layer_forward` calls in each scheme. Checkpointing recomputes every layer's forward exactly once more (the segment replay), regardless of `L` or the segment size -- so the *relative* extra forward compute should be constant, not growing with depth.

In [ ]:
def count_forward_calls(L, m):
    global layer_forward
    orig_forward = layer_forward
    counts = {'n': 0}
    def counting_forward(a_prev, W, b):
        counts['n'] += 1
        return orig_forward(a_prev, W, b)
    layer_forward = counting_forward   # patch the module-level name every helper calls
    try:
        Ws_, bs_ = make_mlp(L, seed=3)
        x_ = rng.normal(size=d)
        go_ = rng.normal(size=d)
        counts['n'] = 0
        forward_backward_no_checkpoint(x_, Ws_, bs_, go_)
        no_ckpt_calls = counts['n']
        counts['n'] = 0
        forward_backward_checkpointed(x_, Ws_, bs_, go_, m)
        ckpt_calls = counts['n']
    finally:
        layer_forward = orig_forward   # always restore, even if an assertion fires above
    return no_ckpt_calls, ckpt_calls

for L_ in [16, 64, 256]:
    m_ = int(np.ceil(np.sqrt(L_)))
    no_calls, ck_calls = count_forward_calls(L_, m_)
    extra_pct = 100 * (ck_calls - no_calls) / no_calls
    print(f'L={L_:4d}: forward calls no-checkpoint={no_calls:4d}  checkpointed={ck_calls:4d}  extra={extra_pct:.1f}%')

Each `layer_forward` call is one unit of forward FLOPs; checkpointing calls it exactly `L` extra times (one replay per layer, split across segments) — a constant **100% extra *forward*** compute, which combined with the (unchanged) backward pass works out to the ~30–40% extra *total* training compute derived on the wiki page ($1 + 1_{\text{recompute}} + 2_{\text{bwd}} = 4$ vs $1 + 2 = 3$ units).

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **extra compute** | ~2× forward passes (demo) — bad when compute-bound |
| **wrong segment size** | too few checkpoints = more recompute; too many = less saving |
| **RNG in forward** | recomputed dropout/noise must reuse the same seed |
| **non-determinism** | recompute must match the original forward exactly (verified) |
| **not automatic gains** | only helps activation memory, not parameter/optimizer memory |

Demo: checkpointing doubles forward compute to save memory.

In [ ]:
# Checkpointing is not free: it RECOMPUTES the forward activations it did not store, so the
# backward pass costs an extra forward pass (~2x forward compute overall). This is the classic
# memory-vs-compute trade-off — you spend compute to fit a bigger model in the same memory. We
# count the forward calls to confirm the overhead.
no_calls, ck_calls = count_forward_calls(256, 16)
print(f'forward calls over training step: no-checkpoint={no_calls}, checkpointed={ck_calls}')
print(f'extra compute: {100 * (ck_calls - no_calls) / no_calls:.0f}%')
assert ck_calls > no_calls, 'checkpointing recomputes activations -> more forward compute for less memory'
print('\nCheckpointing buys memory with compute -> use it when you are memory-bound, not compute-bound.')

## ✏️ Your turn

**Task — Find the memory-optimal segment size empirically.** The wiki page derives $m=\sqrt{L}$ analytically by minimizing `L/m + m`. Fill in `sweep_segment_sizes` below to measure the *actual* peak activation units (using `forward_backward_checkpointed`) for a range of segment sizes `m` at a fixed `L=256`, and confirm the empirical minimum lands at (or very near) `m = round(sqrt(L))`.

In [ ]:
def sweep_segment_sizes(L, m_values):
    """Return a dict {m: peak_units} by calling forward_backward_checkpointed for each m in m_values."""
    Ws_, bs_ = make_mlp(L, seed=4)
    x_ = rng.normal(size=d)
    go_ = rng.normal(size=d)
    results = {}
    # TODO(you): for each m in m_values, run forward_backward_checkpointed(x_, Ws_, bs_, go_, m)
    #            and store the returned peak_units in results[m]
    return results

L_test = 256
m_values = [4, 8, 12, 16, 20, 24, 32, 48, 64]
results = sweep_segment_sizes(L_test, m_values)

if results:
    best_m = min(results, key=results.get)
    print('peak units per segment size:', results)
    print(f'empirical best m = {best_m}, sqrt(L) ~ {np.sqrt(L_test):.1f}')
    assert abs(best_m - np.sqrt(L_test)) <= max(4, 0.25 * np.sqrt(L_test)), \
        'empirical optimum should land near sqrt(L)'
    print('PASS: empirical optimum is close to the analytical sqrt(L) heuristic')

<details><summary>Solution</summary>

```python
def sweep_segment_sizes(L, m_values):
    Ws_, bs_ = make_mlp(L, seed=4)
    x_ = rng.normal(size=d)
    go_ = rng.normal(size=d)
    results = {}
    for m in m_values:
        _, _, _, peak = forward_backward_checkpointed(x_, Ws_, bs_, go_, m)
        results[m] = peak
    return results
```

Plotting `results` against `m_values` reproduces the `L/m + m` curve from the wiki page: it decreases while checkpoints dominate (small `m`), bottoms out near `m = sqrt(L)`, then increases again as the current segment's recomputed activations dominate (large `m`).
</details>

## Key takeaways

- **Store $\sim\sqrt{L}$ checkpoints, recompute the rest:** memory drops from $O(L)$ to
  $O(\sqrt{L})$ (verified).
- **Gradients are identical:** recomputation changes memory, not values (verified).
- **Cost is extra compute:** ~2× forward passes (demo) — a memory-for-compute trade.
- **Use it when memory-bound** (deep nets, long sequences, big batches).